In [ ]:
pip uninstall torch-geometric torch-scatter torch-sparse torch-cluster torch-spline-conv -y

In [ ]:
pip install torch-geometric

In [ ]:
from IPython.core.display import HTML
HTML("<script>Jupyter.notebook.kernel.restart()</script>")

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch_geometric.data import HeteroData
import torch_geometric.transforms as T
import numpy as np
from sklearn.model_selection import train_test_split

In [ ]:
df_fase1=pd.read_csv('/kaggle/input/datasets/nico14h/metagenomica/dataset.csv', sep='\t')
print(df_fase1)

In [ ]:
import torch
from torch_geometric.data import HeteroData
import torch_geometric.transforms as T
import numpy as np

# ==========================================
# 1. INICIALIZAR EL GRAFO (HETERODATA)
# ==========================================
data = HeteroData()
num_pacientes = df_fase1['paciente_id'].max() + 1
num_bacterias = df_fase1['bacteria_id'].max() + 1

# NUEVA LÓGICA: Pacientes como "pizarra en blanco". 
# Les damos un vector neutral de 16 dimensiones llenas de '1s'.
dim_oculta_inicial = 16
data['paciente'].x = torch.ones((num_pacientes, dim_oculta_inicial), dtype=torch.float)

# A las bacterias sí les dejamos el One-Hot, porque funcionan como un "diccionario universal"
data['bacteria'].x = torch.eye(num_bacterias, dtype=torch.float)

# ==========================================
# 2. CONEXIONES (ARISTAS) Y PESOS (CLR)
# ==========================================
df_conexiones_reales = df_fase1[df_fase1['AbundanciaTaxonomica'] != 0.0]
origen = torch.tensor(df_conexiones_reales['paciente_id'].values, dtype=torch.long)
destino = torch.tensor(df_conexiones_reales['bacteria_id'].values, dtype=torch.long)
data['paciente', 'abundancia_de', 'bacteria'].edge_index = torch.stack([origen, destino], dim=0)

# Las abundancias ahora contienen los valores rCLR (positivos y negativos)
data['paciente', 'abundancia_de', 'bacteria'].edge_attr = torch.tensor(
    df_conexiones_reales['AbundanciaTaxonomica'].values, dtype=torch.float
).unsqueeze(1)

num_aristas = data['paciente', 'abundancia_de', 'bacteria'].edge_index.shape[1]
print(f"Cantidad total de aristas en el grafo: {num_aristas}")
# ==========================================
# 3. ETIQUETAS Y MÁSCARAS BLINDADAS
# ==========================================
df_etiquetas = df_fase1[['paciente_id', 'Diagnosticos']].drop_duplicates().sort_values('paciente_id')
tensor_y = torch.zeros(num_pacientes, dtype=torch.long)

for _, fila in df_etiquetas.iterrows():
    tensor_y[int(fila['paciente_id'])] = int(fila['Diagnosticos'])

data['paciente'].y = tensor_y

# Direccionalidad del grafo
data = T.ToUndirected()(data)

print("Grafo inicializado exitosamente con vectores neutrales (Sin PCA).")
print(f"Dimensiones de las características de pacientes (x): {data['paciente'].x.shape}")

In [ ]:
import numpy as np
import torch

# Obtenemos la cantidad total de pacientes
num_pacientes = data['paciente'].num_nodes

# Inicializamos tensores booleanos en False
train_mask = torch.zeros(num_pacientes, dtype=torch.bool)
test_mask = torch.zeros(num_pacientes, dtype=torch.bool)

indices = np.arange(num_pacientes)
etiquetas = data['paciente'].y.numpy()

# Separar los índices según la clase real
indices_hc = indices[etiquetas == 0]
indices_tea = indices[etiquetas == 1]

# ELECCIÓN ALEATORIA EXACTA: 40 HC y 56 TEA
np.random.seed(42) # Semilla para que los resultados sean constantes en tu tesis
train_hc = np.random.choice(indices_hc, size=40, replace=False)
train_tea = np.random.choice(indices_tea, size=56, replace=False)

# Unimos para formar la bolsa de entrenamiento de 96 pacientes
train_idx = np.concatenate([train_hc, train_tea])

# Los pacientes restantes (121 - 96 = 25) van a la bolsa de Test/Validación
test_idx = np.setdiff1d(indices, train_idx)

# Activamos las máscaras
train_mask[train_idx] = True
test_mask[test_idx] = True

data['paciente'].train_mask = train_mask
data['paciente'].test_mask = test_mask

print(f"Bolsa de Entrenamiento: {len(train_idx)} pacientes ({len(train_hc)} HC, {len(train_tea)} TEA)")
print(f"Bolsa de Validación/Test: {len(test_idx)} pacientes")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, HeteroConv

class GAT_Neurodesarrollo(nn.Module):
    def __init__(self, hidden_channels, out_channels, edge_types, num_classes): 
        super().__init__()

        conv1_dict = {}
        conv2_dict = {}
        for edge_type in edge_types:
            # Capa 1: Absorbe la información inicial (Paciente <-> Bacteria)
            conv1_dict[edge_type] = GATConv(
                (-1, -1), hidden_channels, add_self_loops=False, heads=2, concat=False, edge_dim=1
            )
            # Capa 2: Refina el patrón comunitario
            conv2_dict[edge_type] = GATConv(
                (-1, -1), out_channels, add_self_loops=False, heads=1, concat=False, edge_dim=1
            )

        self.conv1 = HeteroConv(conv1_dict, aggr='mean')
        self.conv2 = HeteroConv(conv2_dict, aggr='mean')

        # CONEXIÓN RESIDUAL PARA BACTERIAS:
        # Evita que las bacterias pierdan su identidad One-Hot tras el Salto 1
        self.lin_bacteria = nn.LazyLinear(hidden_channels)

        self.dropout = nn.Dropout(p=0.4)
        
        # CONEXIÓN JUMPING KNOWLEDGE PARA PACIENTES:
        # El clasificador recibe la concatenación del Salto 1 (hidden_channels) + Salto 2 (out_channels)
        self.clasificador = nn.Linear(hidden_channels + out_channels, num_classes)

    def forward(self, x_dict, edge_index_dict, edge_attr_dict):
        # Proyectamos la identidad One-Hot original de las bacterias antes del Salto 1
        identidad_bacteria = self.lin_bacteria(x_dict['bacteria'])

        # --- PRIMER SALTO (1-Hop: Perfil directo Paciente <- Bacteria) ---
        x_dict_1 = self.conv1(x_dict, edge_index_dict, edge_attr_dict=edge_attr_dict)
        
        # Guardamos el perfil de los pacientes del Salto 1
        x_p1 = F.elu(x_dict_1['paciente'])
        
        # A las bacterias les sumamos su identidad residual para que no colapsen a una constante
        x_b1 = F.elu(x_dict_1['bacteria'] + identidad_bacteria)

        # Aplicamos Dropout para regularizar el paso al Salto 2
        x_dict_intermedio = {
            'paciente': self.dropout(x_p1),
            'bacteria': self.dropout(x_b1)
        }

        # --- SEGUNDO SALTO (2-Hop: Contexto comunitario) ---
        x_dict_2 = self.conv2(x_dict_intermedio, edge_index_dict, edge_attr_dict=edge_attr_dict)
        x_p2 = F.elu(x_dict_2['paciente'])

        # --- CLASIFICACIÓN COMBINADA (Salto 1 + Salto 2) ---
        x_pacientes_final = torch.cat([x_p1, x_p2], dim=-1)
        salida = self.clasificador(x_pacientes_final)
        return salida

# Instanciamos con dimensiones moderadas (16 y 8) ahora que el flujo de información está reparado
tipos_aristas = data.metadata()[1]
modelo = GAT_Neurodesarrollo(hidden_channels=8, out_channels=4, edge_types=tipos_aristas, num_classes=2)
print("¡Modelo GAT con Conexiones Residuales instanciado correctamente!")

In [ ]:
import torch.optim as optim
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# ==========================================
# 1. PREPARACIÓN DEL ENTORNO Y DATOS
# ==========================================
# Mandamos el modelo y los datos a la placa de video (GPU) si está disponible
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
modelo = modelo.to(device)
data = data.to(device)

# Aislamos las etiquetas reales y la máscara de entrenamiento ANTES de calcular los pesos
etiquetas_reales = data['paciente'].y
mascara_train = data['paciente'].train_mask

# ==========================================
# 2. INICIALIZACIÓN PEREZOSA Y AJUSTE DE PESOS
# ==========================================
# !!! LA SOLUCIÓN AL ERROR !!!
# Pase de prueba (Dummy pass) para que el modelo construya sus tensores
# antes de enviárselos al optimizador.
with torch.no_grad():
    modelo(data.x_dict, data.edge_index_dict, data.edge_attr_dict)

# Calcula pesos matemáticamente exactos basados en el conjunto de entrenamiento
pesos_calculados = compute_class_weight(
    class_weight='balanced', 
    classes=np.array([0, 1]), 
    y=etiquetas_reales[mascara_train].cpu().numpy()
)
pesos_clases = torch.tensor(pesos_calculados, dtype=torch.float).to(device)

print(f"Pesos calculados automáticamente (HC, TEA): {pesos_calculados}")

# Función de pérdida con los pesos dinámicos inyectados
criterio = nn.CrossEntropyLoss(weight=pesos_clases)

# AHORA SÍ: El optimizador puede recibir los parámetros porque ya están inicializados
optimizador = optim.AdamW(modelo.parameters(), lr=0.0005, weight_decay=5e-4)

# ==========================================
# 3. BUCLE DE ENTRENAMIENTO ESTABILIZADO
# ==========================================
epocas = 300
print("\n--- INICIANDO ENTRENAMIENTO DE GAT (2 CAPAS) ---")

for epoca in range(epocas):
    modelo.train()
    optimizador.zero_grad()

    # Forward Pass (Propagamos TODA la información por el grafo)
    salida = modelo(data.x_dict, data.edge_index_dict, data.edge_attr_dict)

    # Cálculo de Error (Loss) SOLO sobre los pacientes de entrenamiento
    loss = criterio(salida[mascara_train], etiquetas_reales[mascara_train])
    loss.backward()

    # Gradient Clipping: Clava los gradientes para que no exploten
    torch.nn.utils.clip_grad_norm_(modelo.parameters(), max_norm=1.0)

    optimizador.step()

    if (epoca + 1) % 50 == 0:
        print(f'Época {epoca+1:03d} | Pérdida (Loss): {loss.item():.4f}')

# ==========================================
# 4. EVALUACIÓN Y REPORTE VISUAL
# ==========================================
print("\n--- INICIANDO EVALUACIÓN EN CONJUNTO DE PRUEBA ---")
modelo.eval()

with torch.no_grad():
    salida_final = modelo(data.x_dict, data.edge_index_dict, data.edge_attr_dict)
    clases_predichas = salida_final.argmax(dim=1)

mascara_test = data['paciente'].test_mask
y_real = etiquetas_reales[mascara_test].cpu().numpy()
y_predicho = clases_predichas[mascara_test].cpu().numpy()

nombres_clases = ['Clase 0 (HC)', 'Clase 1 (TEA)']

print("\nREPORTE DE CLASIFICACIÓN:")
print(classification_report(y_real, y_predicho, target_names=nombres_clases))

matriz = confusion_matrix(y_real, y_predicho)
plt.figure(figsize=(6, 4))
sns.heatmap(matriz, annot=True, fmt='d', cmap='Blues',
            xticklabels=nombres_clases,
            yticklabels=nombres_clases)
plt.title('Matriz de Confusión - Detección Microbiota (GAT)')
plt.ylabel('Diagnóstico Real')
plt.xlabel('Predicción del Modelo')
plt.show()

In [ ]:
import pandas as pd
import torch

print("--- INICIANDO EXPLICABILIDAD (EXTRACCIÓN DE ATENCIÓN) ---")

# 1. Elegir un paciente para analizar (como antes)
mascara_test = data['paciente'].test_mask
pacientes_test_idx = mascara_test.nonzero(as_tuple=True)[0]

# Tomamos el primer paciente de test como ejemplo
paciente_objetivo = pacientes_test_idx[0].item()
diagnostico_real = data['paciente'].y[paciente_objetivo].item()
print(f"Analizando Paciente ID Interno: {paciente_objetivo} | Diagnóstico real: {diagnostico_real}")

# =========================================================
# EL TRUCO DE LA ATENCIÓN: Extraer los pesos directamente
# =========================================================
# En lugar de usar GNNExplainer, le pedimos a tu capa GATConv
# que nos devuelva exactamente cuánta "atención" (importancia)
# le prestó a cada conexión durante el último salto.

tipo_arista = ('paciente', 'abundancia_de', 'bacteria')
conexiones = data.edge_index_dict[tipo_arista]

modelo.eval()
with torch.no_grad():
    # Accedemos a la capa específica dentro del HeteroConv
    capa_gat = modelo.conv1.convs[tipo_arista]

    # x_tupla: (características de origen, características de destino)
    x_tupla = (data.x_dict['paciente'], data.x_dict['bacteria'])

    # Hacemos una pasada por esta única capa pidiéndole que DEVUELVA LOS PESOS DE ATENCIÓN (return_attention_weights=True)
    out, (edge_index_out, alpha) = capa_gat(
        x_tupla,
        conexiones,
        edge_attr=data.edge_attr_dict[tipo_arista],
        return_attention_weights=True
    )

# 'alpha' contiene la importancia (de 0.0 a 1.0) que la GAT le dio a CADA arista
# Promediamos si hubieras usado múltiples heads (como usaste heads=1, no cambia nada, pero es buena práctica)
if alpha.dim() > 1:
    importancia_aristas = alpha.mean(dim=-1).squeeze()
else:
    importancia_aristas = alpha.squeeze()


# =========================================================
# FILTRAR PARA EL PACIENTE OBJETIVO (Mismo código de antes)
# =========================================================
# La fila 0 de las conexiones son los pacientes, la fila 1 son las bacterias
aristas_del_paciente = (conexiones[0] == paciente_objetivo)
bacterias_conectadas = conexiones[1][aristas_del_paciente]
importancia_bacterias = importancia_aristas[aristas_del_paciente]

# Armar el ranking con las bacterias más importantes
ranking = pd.DataFrame({
    'bacteria_idx_interno': bacterias_conectadas.cpu().numpy(),
    'importancia_GNN': importancia_bacterias.cpu().numpy()
})

# Ordenamos de mayor a menor importancia
ranking = ranking.sort_values(by='importancia_GNN', ascending=False)

# Escalar para que el máximo sea 1.0 (facilita la visualización luego)
if ranking['importancia_GNN'].max() > 0:
    ranking['importancia_GNN'] = ranking['importancia_GNN'] / ranking['importancia_GNN'].max()

nombres_clases = ['Clase 0 (HC)', 'Clase 1 (TEA)']
print(f"\nTOP 5 BACTERIAS MÁS IMPORTANTES PARA DIAGNOSTICAR {nombres_clases[diagnostico_real]}:")
print(ranking.head(5).to_string(index=False))

In [ ]:
import pandas as pd
import torch

print("--- INICIANDO EXPLICABILIDAD (EXTRACCIÓN DE ATENCIÓN) ---")

# 1. Elegir un paciente para analizar (como antes)
mascara_test = data['paciente'].test_mask
pacientes_test_idx = mascara_test.nonzero(as_tuple=True)[0]

# Tomamos el primer paciente de test como ejemplo
paciente_objetivo = pacientes_test_idx[10].item()
diagnostico_real = data['paciente'].y[paciente_objetivo].item()
print(f"Analizando Paciente ID Interno: {paciente_objetivo} | Diagnóstico real: {diagnostico_real}")

# =========================================================
# EL TRUCO DE LA ATENCIÓN: Extraer los pesos directamente
# =========================================================
# En lugar de usar GNNExplainer, le pedimos a tu capa GATConv
# que nos devuelva exactamente cuánta "atención" (importancia)
# le prestó a cada conexión durante el último salto.

tipo_arista = ('paciente', 'abundancia_de', 'bacteria')
conexiones = data.edge_index_dict[tipo_arista]

modelo.eval()
with torch.no_grad():
    # Accedemos a la capa específica dentro del HeteroConv
    capa_gat = modelo.conv1.convs[tipo_arista]

    # x_tupla: (características de origen, características de destino)
    x_tupla = (data.x_dict['paciente'], data.x_dict['bacteria'])

    # Hacemos una pasada por esta única capa pidiéndole que DEVUELVA LOS PESOS DE ATENCIÓN (return_attention_weights=True)
    out, (edge_index_out, alpha) = capa_gat(
        x_tupla,
        conexiones,
        edge_attr=data.edge_attr_dict[tipo_arista],
        return_attention_weights=True
    )

# 'alpha' contiene la importancia (de 0.0 a 1.0) que la GAT le dio a CADA arista
# Promediamos si hubieras usado múltiples heads (como usaste heads=1, no cambia nada, pero es buena práctica)
if alpha.dim() > 1:
    importancia_aristas = alpha.mean(dim=-1).squeeze()
else:
    importancia_aristas = alpha.squeeze()


# =========================================================
# FILTRAR PARA EL PACIENTE OBJETIVO (Mismo código de antes)
# =========================================================
# La fila 0 de las conexiones son los pacientes, la fila 1 son las bacterias
aristas_del_paciente = (conexiones[0] == paciente_objetivo)
bacterias_conectadas = conexiones[1][aristas_del_paciente]
importancia_bacterias = importancia_aristas[aristas_del_paciente]

# Armar el ranking con las bacterias más importantes
ranking = pd.DataFrame({
    'bacteria_idx_interno': bacterias_conectadas.cpu().numpy(),
    'importancia_GNN': importancia_bacterias.cpu().numpy()
})

# Ordenamos de mayor a menor importancia
ranking = ranking.sort_values(by='importancia_GNN', ascending=False)

# Escalar para que el máximo sea 1.0 (facilita la visualización luego)
if ranking['importancia_GNN'].max() > 0:
    ranking['importancia_GNN'] = ranking['importancia_GNN'] / ranking['importancia_GNN'].max()

nombres_clases = ['Clase 0 (HC)', 'Clase 1 (TEA)']
print(f"\nTOP 5 BACTERIAS MÁS IMPORTANTES PARA DIAGNOSTICAR {nombres_clases[diagnostico_real]}:")
print(ranking.head(5).to_string(index=False))

In [ ]:
import pandas as pd

# 1. Hacemos el factorize, pero guardamos el resultado completo de las bacterias
factorizacion_pacientes = pd.factorize(df_fase1['paciente_id'])
factorizacion_bacterias = pd.factorize(df_fase1['bacteria_id']) # Aquí 'bacteria_id' debe tener los nombres crudos

# 2. Asignamos los números al DataFrame como ya venías haciendo
df_fase1['paciente_id'] = factorizacion_pacientes[0]
df_fase1['bacteria_id'] = factorizacion_bacterias[0]

# 3. CREAMOS EL DICCIONARIO INVERSO MÁGICO (De Número -> Nombre)
# factorizacion_bacterias[1] contiene los nombres originales en el orden exacto de los IDs generados
diccionarioBacteriasDos = {indice: nombre for indice, nombre in enumerate(factorizacion_bacterias[1])}

print("Diccionario de bacterias creado con éxito.")

In [ ]:
# =================================================================
# 1. CREACIÓN DEL DICCIONARIO DIRECTAMENTE DESDE EL DATAFRAME
# =================================================================
# Extraemos una tablita que solo tenga el ID interno nuevo y el nombre de texto
df_bacterias_unicas = df_fase1[['bacteria_id', 'Bacteria_Limpia']].drop_duplicates()

# Convertimos esas dos columnas en el diccionario mágico: {ID_interno : 'Nombre Biológico'}
diccionarioBacteriasDos = dict(zip(df_bacterias_unicas['bacteria_id'], df_bacterias_unicas['Bacteria_Limpia']))

# =================================================================
# 2. TRADUCCIÓN DEL RANKING
# =================================================================
# Aplicamos el diccionario a la columna del ranking
ranking['Nombre_Biologico'] = ranking['bacteria_idx_interno'].map(diccionarioBacteriasDos)

# =================================================================
# 3. IMPRIMIR EL TOP 5 TRADUCIDO
# =================================================================
print("\nTOP 5 BACTERIAS MÁS IMPORTANTES PARA DIAGNOSTICAR Clase 1 (TEA):")
columnas_mostrar = ['bacteria_idx_interno', 'Nombre_Biologico', 'importancia_GNN']
print(ranking[columnas_mostrar].head(100).to_string(index=False))

In [ ]:
import torch.nn.functional as F
from torch_geometric.nn import GraphConv, HeteroConv
import torch.nn as nn

class GraphConv_Neurodesarrollo(nn.Module):
    def __init__(self, hidden_channels, out_channels, edge_types, num_classes):
        super().__init__()

        conv1_dict = {}
        conv2_dict = {}

        for edge_type in edge_types:
            conv1_dict[edge_type] = GraphConv((-1, -1), hidden_channels)
            conv2_dict[edge_type] = GraphConv((-1, -1), out_channels)

        # EL CAMBIO CRÍTICO: Reemplazamos aggr='sum' por aggr='mean'
        # Esto normaliza la señal sin importar cuántas bacterias tenga el paciente
        self.conv1 = HeteroConv(conv1_dict, aggr='mean')
        self.conv2 = HeteroConv(conv2_dict, aggr='mean')

        self.dropout = nn.Dropout(p=0.4)
        self.clasificador = nn.Linear(out_channels, num_classes)

    def forward(self, x_dict, edge_index_dict, edge_attr_dict):
        pesos_1D = {key: tensor.view(-1) for key, tensor in edge_attr_dict.items()}

        x_dict = self.conv1(x_dict, edge_index_dict, edge_weight_dict=pesos_1D)
        x_dict = {key: F.elu(x) for key, x in x_dict.items()}
        x_dict = {key: self.dropout(x) for key, x in x_dict.items()}

        x_dict = self.conv2(x_dict, edge_index_dict, edge_weight_dict=pesos_1D)

        x_pacientes = x_dict['paciente']
        salida = self.clasificador(x_pacientes)
        return salida

# Instanciamos con baja complejidad para evitar el sobreajuste
tipos_aristas = data.metadata()[1]
modelo = GraphConv_Neurodesarrollo(hidden_channels=16, out_channels=8, edge_types=tipos_aristas, num_classes=2)
print("¡Modelo GraphConv reinstanciado con agregación por Promedio (mean)!")

In [ ]:
import torch.optim as optim
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# ==========================================
# 1. PREPARACIÓN Y NUEVOS HIPERPARÁMETROS
# ==========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
modelo = modelo.to(device)
data = data.to(device)

# Reducimos el Learning Rate a 0.001 (crucial para la estabilidad)
# Y bajamos el weight_decay a 1e-4 para que no pelee contra el gradiente
optimizador = optim.AdamW(modelo.parameters(), lr=0.001, weight_decay=1e-4)

# Castigo Asimétrico Severo: Obligamos a la red a prestar atención a la clase HC (0)
# Le damos el doble de peso matemático a la clase 0
pesos_agresivos = torch.tensor([2.0, 1.0], dtype=torch.float).to(device)
criterio = nn.CrossEntropyLoss(weight=pesos_agresivos)

# ==========================================
# 2. BUCLE DE ENTRENAMIENTO ESTABILIZADO
# ==========================================
epocas = 120 # Podemos darle más épocas ahora que avanza más lento
print("--- INICIANDO ENTRENAMIENTO ESTABILIZADO ---")

for epoca in range(epocas):
    modelo.train()
    optimizador.zero_grad()

    salida = modelo(data.x_dict, data.edge_index_dict, data.edge_attr_dict)

    predicciones = salida
    etiquetas_reales = data['paciente'].y
    mascara_train = data['paciente'].train_mask

    loss = criterio(predicciones[mascara_train], etiquetas_reales[mascara_train])
    loss.backward()

    # EL SECRETO DE LA ESTABILIDAD: Gradient Clipping
    # Evita que las actualizaciones de los pesos exploten o reboten violentamente
    torch.nn.utils.clip_grad_norm_(modelo.parameters(), max_norm=1.0)

    optimizador.step()

    # Imprimimos el progreso
    if (epoca + 1) % 20 == 0:
        print(f'Época {epoca+1:03d} | Pérdida (Loss): {loss.item():.4f}')

# ==========================================
# 3. EVALUACIÓN Y REPORTE
# ==========================================
print("\n--- INICIANDO EVALUACIÓN EN CONJUNTO DE PRUEBA ---")
modelo.eval()

with torch.no_grad():
    salida_final = modelo(data.x_dict, data.edge_index_dict, data.edge_attr_dict)
    clases_predichas = salida_final.argmax(dim=1)

mascara_test = data['paciente'].test_mask
y_real = data['paciente'].y[mascara_test].cpu().numpy()
y_predicho = clases_predichas[mascara_test].cpu().numpy()

nombres_clases = ['Clase 0 (HC)', 'Clase 1 (TEA)']

print("\nREPORTE DE CLASIFICACIÓN:")
print(classification_report(y_real, y_predicho, target_names=nombres_clases))

matriz = confusion_matrix(y_real, y_predicho)
plt.figure(figsize=(6, 4))
sns.heatmap(matriz, annot=True, fmt='d', cmap='Blues',
            xticklabels=nombres_clases,
            yticklabels=nombres_clases)
plt.title('Matriz de Confusión - Detección Microbiota (GraphConv)')
plt.ylabel('Diagnóstico Real')
plt.xlabel('Predicción del Modelo')
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import torch

print("--- EXTRACCIÓN BIOLÓGICA CON GRAPHCONV (MÉTODO DE GRADIENTES) ---")

# 1. Recuperamos el diccionario biológico de bacterias
df_bacterias = df_fase1[['bacteria_id', 'Bacteria_Limpia']].drop_duplicates()
diccionario_nombres = dict(zip(df_bacterias['bacteria_id'], df_bacterias['Bacteria_Limpia']))

# 2. Preparamos el grafo para calcular derivadas sobre la abundancia
modelo.eval()
tipo_arista = ('paciente', 'abundancia_de', 'bacteria')

# Clonamos la columna de abundancia y le ordenamos a PyTorch que rastree sus gradientes
abundancias_originales = data.edge_attr_dict[tipo_arista].clone().detach()
abundancias_originales.requires_grad_(True)

# Inyectamos esta columna rastreable temporalmente en el diccionario de aristas
edge_attr_dict_expl = data.edge_attr_dict.copy()
edge_attr_dict_expl[tipo_arista] = abundancias_originales

# 3. Hacemos un Forward Pass (Diagnóstico)
salida = modelo(data.x_dict, data.edge_index_dict, edge_attr_dict_expl)

# 4. Aislamos a los pacientes que REALMENTE tienen TEA (Clase 1) en el conjunto de prueba
mascara_test = data['paciente'].test_mask
etiquetas = data['paciente'].y
pacientes_tea_test = (mascara_test == True) & (etiquetas == 1)

# Sumamos cuánta "seguridad matemática" tuvo la red de que estos pacientes eran TEA (Columna 1)
score_total_tea = salida[pacientes_tea_test, 1].sum()

# 5. EL TRUCO MATEMÁTICO: Retropropagación (Backward)
score_total_tea.backward()

# Aplicamos la fórmula Input x Gradient y usamos .detach() antes de pasarlo a NumPy
importancia_global = (abundancias_originales.grad.abs() * abundancias_originales.abs()).squeeze().detach().cpu().numpy()

# 6. Filtramos y mapeamos la importancia a cada bacteria
conexiones = data.edge_index_dict[tipo_arista].cpu().numpy()
nodos_origen_pacientes = conexiones[0]
nodos_destino_bacterias = conexiones[1]

ids_pacientes = torch.where(pacientes_tea_test)[0].cpu().numpy()
ranking_bacterias = {}

for paciente_id in ids_pacientes:
    # Aislamos los "puentes" de este paciente en particular
    aristas_del_paciente = (nodos_origen_pacientes == paciente_id)
    bacterias = nodos_destino_bacterias[aristas_del_paciente]
    importancia = importancia_global[aristas_del_paciente]

    # Acumulamos la importancia matemática de sus bacterias
    for bac, imp in zip(bacterias, importancia):
        if bac not in ranking_bacterias:
            ranking_bacterias[bac] = 0
        ranking_bacterias[bac] += imp

# 7. Armamos el reporte clínico (Top 15)
df_ranking = pd.DataFrame(list(ranking_bacterias.items()), columns=['ID_Interno', 'Score_Importancia'])
df_ranking = df_ranking.sort_values(by='Score_Importancia', ascending=False).head(15)

# Traducimos de ID a Nombre Biológico
df_ranking['Bacteria'] = df_ranking['ID_Interno'].map(diccionario_nombres)

# Normalizamos del 0% al 100% para mejor visualización
max_score = df_ranking['Score_Importancia'].max()
if max_score > 0:
    df_ranking['Relevancia_Relativa (%)'] = (df_ranking['Score_Importancia'] / max_score) * 100
else:
    df_ranking['Relevancia_Relativa (%)'] = 0

print("🏆 TOP 15 BACTERIAS CLAVE PARA DETECTAR TEA (Método Input x Gradient):")
print(df_ranking[['Bacteria', 'Relevancia_Relativa (%)']].to_string(index=False, float_format="%.2f"))